# Export to Android

CRISP-DM phase: Deployment

## 1. Setup and Data Loading

The chosen model for the app is the logistic regression model (see `ModelComparison.ipynb`). This notebook turns it into a Kotlin file that the Android app can use without Python or any library, and creates test cases that check the app gives the same answers as the Python model.

### 1.1 Importing Libraries

In [1]:
#necessary imports
import os
import joblib
import numpy as np
import pandas as pd

### 1.2 Loading the Model and the Data

In [2]:
#loading the saved logistic regression model
model_lr = joblib.load('../models/logistic_regression_model.pkl')

In [3]:
#loading the saved training and test sets
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
X_train = train.drop('level', axis=1)
X_test = test.drop('level', axis=1)
y_test = test['level']

In [4]:
#all students together (training and test), to check the calculation on every student
X_all = pd.concat([X_train, X_test])
print(X_all.shape)

(19251, 21)


## 2. What the Model Contains

A logistic regression model with three levels stores one weight for every question and level, and one intercept for every level. For one student, the score of a level is its intercept plus the weight of each question multiplied by the student's answer, added up over the 21 questions. The level with the highest score is the prediction. The scores can also be turned into probabilities that add up to 1 (a softmax). The model was trained on the raw answers with no scaling, so nothing else is needed.

In [5]:
#the levels, the shape of the weights and the intercepts
print(model_lr.classes_)
print(model_lr.coef_.shape)
print(model_lr.intercept_)

[0 1 2]
(3, 21)
[ 5.92995049  1.20992053 -7.13987102]


In [6]:
#the order of the questions the model expects
print(list(X_train.columns))

['phq_q2', 'phq_q1', 'gad_q6', 'gad_q2', 'gad_q3', 'gad_q4', 'gad_q1', 'pss_q2', 'pss_q3', 'pss_q1', 'gad_q5', 'gad_q7', 'isi_q1', 'isi_q7', 'isi_q2', 'pss_q14', 'isi_q4', 'pss_q11', 'isi_q5', 'isi_q3', 'pss_q8']


#### Observation

The model has the levels 0, 1 and 2 in that order, a weight matrix of 3 levels by 21 questions (63 weights) and 3 intercepts, so the whole model is 66 numbers. The question order printed above is the order of the columns in the weights. The Kotlin file keeps the question names in `featureNames` and looks each answer up by its question id, so the app does not have to send the answers in this order.

## 3. Answer Ranges

The model only knows the answers on the scale used in the training data. If the app coded an answer differently (for example 1 to 4 instead of 0 to 3), the predictions would be wrong without any error. The lowest and highest answer of each question in the data are therefore built into the Kotlin file as a check.

In [7]:
#lowest and highest answer of each question in all the data
ranges = pd.DataFrame({'lowest answer': X_all.min(), 'highest answer': X_all.max()})
ranges.T

,phq_q2,phq_q1,gad_q6,gad_q2,gad_q3,gad_q4,gad_q1,pss_q2,pss_q3,pss_q1,...,gad_q7,isi_q1,isi_q7,isi_q2,pss_q14,isi_q4,pss_q11,isi_q5,isi_q3,pss_q8
lowest answer,0,0,0,0,0,0,0,1,1,1,...,0,0,0,0,1,0,1,0,0,1
highest answer,3,3,3,3,3,3,3,5,5,5,...,3,4,4,4,5,4,5,4,4,5


#### Observation

The PHQ and GAD questions are coded 0 to 3, the six ISI questions 0 to 4 and the six PSS questions 1 to 5. The PSS answers start at 1 and not at 0 as in the usual scoring of the PSS-14. The app must send the answers on exactly these scales, and the PSS answers must be numbered 1 to 5. If the app numbered them 0 to 4, every PSS answer would be one too low and the predictions would shift. The Kotlin file rejects answers outside these ranges, which catches this kind of mistake for the lowest and highest answer, but it cannot catch a shift that stays inside the range, so the answer scales in the app should be checked against this table.

## 4. Checking the Calculation in Plain Python

The Kotlin code will do the same simple loops, so they are written out here in plain Python first and compared with the predictions of the scikit-learn model for every student.

In [8]:
#the weights and intercepts as plain lists of numbers
weights = model_lr.coef_.tolist()
intercepts = model_lr.intercept_.tolist()

In [9]:
#scores of the three levels for every student, calculated by hand in the same order as the kotlin code
all_scores = []
for row in X_all.to_numpy().tolist():
    scores = []
    for k in range(3):
        total = intercepts[k]
        for j in range(21):
            total += weights[k][j] * row[j]
        scores.append(total)
    all_scores.append(scores)

In [10]:
#predicted level = the level with the highest score
by_hand = [scores.index(max(scores)) for scores in all_scores]
print((np.array(by_hand) != model_lr.predict(X_all)).sum(), 'students with a different prediction out of', len(X_all))

0 students with a different prediction out of 19251


In [11]:
#probabilities by hand (softmax) compared with the probabilities of the model
s = np.array(all_scores)
exps = np.exp(s - s.max(axis=1, keepdims=True))
by_hand_proba = exps / exps.sum(axis=1, keepdims=True)
print(np.abs(by_hand_proba - model_lr.predict_proba(X_all)).max())

1.0547118733938987e-15


#### Observation

The calculation written by hand gives a different prediction from the scikit-learn model for 0 of the 19,251 students, and its probabilities differ from the model's by at most about 1e-15, which is rounding error. The Kotlin code uses the same steps in the same order, so it should give the same answers.

## 5. Writing the Kotlin File

The Kotlin file is made from a template, with the question names, answer ranges, weights and intercepts filled in. The numbers are written with full precision. Change `package_name` to the package of the app before copying the file into the project.

In [12]:
#package name used at the top of the kotlin files (change it to the package of the app)
package_name = 'com.example.screening'

In [13]:
#kotlin template: the placeholders in capital letters are filled in by the next cell
kotlin_template = '''package __PACKAGE__

// The three levels the screening can return.
enum class Level { OKAY, MILD, MODERATE_OR_WORSE }

// Logistic regression model exported from ExportToAndroid.ipynb. Do not edit the numbers by hand.
// Every question has a weight for each level. A level's score is its intercept plus
// (weight x answer) added up over the 21 questions, and the level with the highest score wins.
object LevelClassifier {

    // question ids, in the same order the model was trained on
    val featureNames: List<String> = listOf(__NAMES__)

    // lowest and highest answer seen in the training data for each question (same order as featureNames)
    private val lowestAnswers = intArrayOf(__MINS__)
    private val highestAnswers = intArrayOf(__MAXS__)

    // one row of 21 weights for each level: okay, mild, moderate or worse
    private val weights = arrayOf(
__WEIGHTS__    )

    // one intercept for each level
    private val intercepts = doubleArrayOf(__INTERCEPTS__)

    // the score of each level for one student; answers maps a question id to the answer given
    fun scores(answers: Map<String, Int>): DoubleArray {
        val x = IntArray(featureNames.size)
        for (j in featureNames.indices) {
            val name = featureNames[j]
            val value = answers[name] ?: throw IllegalArgumentException("Missing answer for $name")
            require(value in lowestAnswers[j]..highestAnswers[j]) {
                "Answer for $name must be between ${lowestAnswers[j]} and ${highestAnswers[j]}, but was $value"
            }
            x[j] = value
        }
        val result = DoubleArray(intercepts.size)
        for (k in intercepts.indices) {
            var total = intercepts[k]
            for (j in x.indices) {
                total += weights[k][j] * x[j]
            }
            result[k] = total
        }
        return result
    }

    // the score of each level turned into probabilities that add up to 1
    fun probabilities(answers: Map<String, Int>): DoubleArray {
        val s = scores(answers)
        var highest = s[0]
        for (k in 1 until s.size) {
            if (s[k] > highest) highest = s[k]
        }
        val exps = DoubleArray(s.size)
        var sum = 0.0
        for (k in s.indices) {
            exps[k] = Math.exp(s[k] - highest)
            sum += exps[k]
        }
        for (k in exps.indices) {
            exps[k] = exps[k] / sum
        }
        return exps
    }

    // the level with the highest score
    fun predictLevel(answers: Map<String, Int>): Level {
        val s = scores(answers)
        var best = 0
        for (k in 1 until s.size) {
            if (s[k] > s[best]) best = k
        }
        return Level.values()[best]
    }
}
'''

In [14]:
#the pieces that go into the template
names_text = ', '.join('"' + name + '"' for name in X_train.columns)
mins_text = ', '.join(str(v) for v in X_all.min())
maxs_text = ', '.join(str(v) for v in X_all.max())
weights_text = ''
for row in weights:
    weights_text += '        doubleArrayOf(' + ', '.join(repr(w) for w in row) + '),\n'
intercepts_text = ', '.join(repr(b) for b in intercepts)

In [15]:
#filling in the template and writing the kotlin file
kotlin_code = kotlin_template.replace('__PACKAGE__', package_name).replace('__NAMES__', names_text)
kotlin_code = kotlin_code.replace('__MINS__', mins_text).replace('__MAXS__', maxs_text)
kotlin_code = kotlin_code.replace('__WEIGHTS__', weights_text).replace('__INTERCEPTS__', intercepts_text)
os.makedirs('../export/android', exist_ok=True)
with open('../export/android/LevelClassifier.kt', 'w', encoding='utf-8', newline='\n') as f:
    f.write(kotlin_code)

In [16]:
#the part of the file that holds the numbers
print(kotlin_code[kotlin_code.index('    // question ids'):kotlin_code.index('    // the score of each level for one student')])

    // question ids, in the same order the model was trained on
    val featureNames: List<String> = listOf("phq_q2", "phq_q1", "gad_q6", "gad_q2", "gad_q3", "gad_q4", "gad_q1", "pss_q2", "pss_q3", "pss_q1", "gad_q5", "gad_q7", "isi_q1", "isi_q7", "isi_q2", "pss_q14", "isi_q4", "pss_q11", "isi_q5", "isi_q3", "pss_q8")

    // lowest and highest answer seen in the training data for each question (same order as featureNames)
    private val lowestAnswers = intArrayOf(0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1)
    private val highestAnswers = intArrayOf(3, 3, 3, 3, 3, 3, 3, 5, 5, 5, 3, 3, 4, 4, 4, 5, 4, 5, 4, 4, 5)

    // one row of 21 weights for each level: okay, mild, moderate or worse
    private val weights = arrayOf(
        doubleArrayOf(-2.0107861012238213, -1.6993804585747831, -0.41052578033928555, -0.2734836931694442, -0.13819657900600268, -0.22138719634828363, -0.2549132039222899, -0.3252579193986125, -0.007904456878753238, 0.04461633806771616, -0.396649801

#### Observation

The Kotlin file `../export/android/LevelClassifier.kt` holds the 21 question names, the answer ranges, the 63 weights and the 3 intercepts as plain numbers, written with full precision, plus about 50 lines of code that add up the scores, convert them to probabilities and return the level. It uses no library, so it works offline on the phone.

## 6. Test Cases

The test file checks that the Kotlin code gives the same level and the same probabilities as the Python model for 30 test students: 8 from each level, plus the 6 students whose two highest scores are closest, which are the ones most likely to be predicted differently by a small rounding difference. The expected values come from the scikit-learn model, not from the hand calculation.

In [17]:
#8 random test students from each level
rng = np.random.default_rng(42)
chosen = []
for level in [0, 1, 2]:
    rows = np.where(y_test == level)[0]
    chosen += rng.choice(rows, size=8, replace=False).tolist()

In [18]:
#the 6 other test students whose two highest scores are closest together
test_scores = np.sort(model_lr.decision_function(X_test), axis=1)
gap = test_scores[:, -1] - test_scores[:, -2]
closest = [i for i in np.argsort(gap) if i not in chosen][:6]
chosen += closest
print(len(chosen), 'test students chosen')
print(gap[closest].round(4))

30 test students chosen
[0.0009 0.0024 0.0047 0.0063 0.0063 0.0081]


In [19]:
#expected level and probabilities from the python model for the chosen students
level_words = ['OKAY', 'MILD', 'MODERATE_OR_WORSE']
expected_level = model_lr.predict(X_test)
expected_proba = model_lr.predict_proba(X_test)
cases_text = ''
for i in chosen:
    answers = ', '.join(str(a) for a in X_test.iloc[i])
    probas = ', '.join(repr(float(p)) for p in expected_proba[i])
    cases_text += '        Case(listOf(' + answers + '), Level.' + level_words[expected_level[i]] + ', doubleArrayOf(' + probas + ')),\n'

In [20]:
#test file template
test_template = '''package __PACKAGE__

import org.junit.Assert.assertArrayEquals
import org.junit.Assert.assertEquals
import org.junit.Test

// Test cases made by ExportToAndroid.ipynb. The expected levels and probabilities come from the Python model.
class LevelClassifierTest {

    private class Case(val answers: List<Int>, val level: Level, val probabilities: DoubleArray)

    private val cases = listOf(
__CASES__    )

    private fun toMap(answers: List<Int>): Map<String, Int> = LevelClassifier.featureNames.zip(answers).toMap()

    @Test
    fun levelsAndProbabilitiesMatchPython() {
        for (case in cases) {
            val answers = toMap(case.answers)
            assertEquals(case.level, LevelClassifier.predictLevel(answers))
            assertArrayEquals(case.probabilities, LevelClassifier.probabilities(answers), 1e-9)
        }
    }

    @Test(expected = IllegalArgumentException::class)
    fun missingAnswerIsRejected() {
        val answers = toMap(cases[0].answers).toMutableMap()
        answers.remove(LevelClassifier.featureNames[0])
        LevelClassifier.predictLevel(answers)
    }

    @Test(expected = IllegalArgumentException::class)
    fun answerOutsideTheTrainingRangeIsRejected() {
        val answers = toMap(cases[0].answers).toMutableMap()
        answers[LevelClassifier.featureNames[0]] = 99
        LevelClassifier.predictLevel(answers)
    }
}
'''

In [21]:
#filling in the test template and writing the test file
test_code = test_template.replace('__PACKAGE__', package_name).replace('__CASES__', cases_text)
with open('../export/android/LevelClassifierTest.kt', 'w', encoding='utf-8', newline='\n') as f:
    f.write(test_code)

In [22]:
#the levels the chosen students are expected to get
print(pd.Series([level_words[expected_level[i]] for i in chosen]).value_counts())

MILD                 12
OKAY                 11
MODERATE_OR_WORSE     7
Name: count, dtype: int64


#### Observation

The 30 chosen students cover all three levels (11 okay, 12 mild and 7 moderate or worse). The 6 closest cases have a gap of less than 0.01 between their two highest scores, so a small calculation difference in the app would be most likely to change their level, which makes them good tests. The test file `../export/android/LevelClassifierTest.kt` also checks that a missing answer and an answer outside the range are rejected.

## 7. Using the Files in Android Studio

1. Copy `LevelClassifier.kt` into the app's source folder (`app/src/main/java/` in the folder of the package) and `LevelClassifierTest.kt` into the test folder (`app/src/test/java/` in the same package folder). The test file needs `testImplementation("junit:junit:4.13.2")`, which new Android Studio projects include by default. If the package of the app is not `com.example.screening`, change `package_name` in Section 5 and run this notebook again, or change the first line of both files.
2. In the app, collect the 21 answers in a map from question id to answer and call the model. The PSS answers must be numbered 1 to 5.

```kotlin
val answers = mapOf("phq_q1" to 2, "phq_q2" to 1, /* ... all 21 questions ... */ "pss_q8" to 4)
when (LevelClassifier.predictLevel(answers)) {
    Level.OKAY -> { /* short message with optional resources */ }
    Level.MILD -> { /* in-app resources, with an easy way to contact a counselor */ }
    Level.MODERATE_OR_WORSE -> { /* ask the student to see a counselor as soon as possible, with support resources for the wait */ }
}
```

3. Run `LevelClassifierTest`. If it passes, the app gives the same answers as the Python model for the 30 test students.
4. If the model is ever trained again, run this notebook again and replace both files, so the weights and the test cases always match.

**Check done outside this notebook:** the two files were compiled with the Kotlin compiler that comes with Android Studio and run with JUnit 4.13.2, and all 3 tests passed. The compiled `LevelClassifier` was also run on all 19,251 students (training and test) and gave a different level from the Python model for 0 of them. This was run on a computer and not on a phone.

**Limits:** the file only checks that the app calculates the same numbers as Python. It does not check that the app asks the questions with the same wording and scales as the data, or that the model works well for new students. The model also never sees PHQ-9 question 9 (thoughts of self-harm), so the app should ask it separately and handle the answer outside the model.